# Phase 2 Fine-Tuning for Speculative MTP Models (CP / HMM)

This notebook simulates Phase 2 (joint training of the target probabilistic head and backbone LoRA adapter) starting from the reference Feed-Forward (FF) checkpoint `_v1`.

It takes an arbitrary models folder path (defaulting to `saved_models`), loads the FF `_v1` checkpoint, swaps the head to the target head (CP or HMM), transfers the weights, and executes the joint fine-tuning loop.

In [1]:
import sys
sys.modules['torchvision'] = None
import os

# Disable Hugging Face telemetry and warnings to avoid hangs
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_EXPERIMENTAL_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Define parameters (can be overridden)
MODELS_DIR = "../saved_models"  # default path to saved_models relative to src/
HEAD_TYPE = "cp"                # "cp" or "hmm"
WINDOW_SIZE = 6                 # speculative window size
RANKS = 32                      # number of components/ranks
BATCH_SIZE = 2                  # batch size
MAX_SAMPLES = 10                # number of samples for training
MAX_LEN = 2048                  # max sequence length
MODEL_ID = "google/byt5-small"
EPOCHS = 1                      # number of epochs to run
LR_HEAD = 3e-4                  # learning rate for the speculative head
LR_BACKBONE = 2e-5              # learning rate for backbone LoRA
WARMUP_STEPS = 5                # linear warmup steps for backbone LoRA

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from tqdm import tqdm
from transformers import AutoTokenizer

# Adjust paths to import from local files in src
sys.path.append(os.path.abspath("."))

from models.probabilistic_circuits import CanonicPolyidiac, MTPC_HMM, FF
from models.mtp_llm import MTP_LLM
from utils import (
    MTPChatDataset, 
    compute_mtpc_loss, 
    get_byt5_preprocess_function, 
    load_tulu_dataset, 
    CHAT_TEMPLATE
)
from training import swap_model_head, init_cp_from_ff

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [3]:
# 1. Locate reference files ending in _v1
ff_head_filename = f"mtp_head_ff_w{WINDOW_SIZE}_phase1_v1.pth"
lora_dir_name = f"lora_ff_w{WINDOW_SIZE}_phase1_v1"

ff_head_path = os.path.join(MODELS_DIR, ff_head_filename)
lora_path = os.path.join(MODELS_DIR, lora_dir_name)

print(f"Searching for FF head weights at: {ff_head_path}")
print(f"Searching for LoRA adapter at: {lora_path}")

if not os.path.exists(ff_head_path):
    # Try absolute path if relative path doesn't exist
    ff_head_path = os.path.abspath(os.path.join(os.getcwd(), MODELS_DIR, ff_head_filename))
    lora_path = os.path.abspath(os.path.join(os.getcwd(), MODELS_DIR, lora_dir_name))
    print(f"Falling back to absolute paths:\n - Head: {ff_head_path}\n - LoRA: {lora_path}")

assert os.path.exists(ff_head_path), f"FF head path not found: {ff_head_path}"
assert os.path.exists(lora_path), f"LoRA adapter path not found: {lora_path}"

Searching for FF head weights at: ../saved_models/mtp_head_ff_w6_phase1_v1.pth
Searching for LoRA adapter at: ../saved_models/lora_ff_w6_phase1_v1


In [4]:
print("Setting up dataset...")
from datasets import load_from_disk

# Search directories for pre-tokenized cached datasets
possible_paths = [
    os.path.join(MODELS_DIR, f"tokenized_train_data_{MAX_SAMPLES}_len{MAX_LEN}"),
    os.path.join(MODELS_DIR, "tokenized_train_data_10_len2048"),
    os.path.join(MODELS_DIR, "tokenized_train_data_100_len2048"),
    os.path.join(MODELS_DIR, "../legacy_models", f"tokenized_train_data_{MAX_SAMPLES}_len{MAX_LEN}"),
    os.path.join(MODELS_DIR, "../legacy_models", "tokenized_train_data_10_len2048"),
    os.path.join(MODELS_DIR, "../legacy_models", "tokenized_train_data_100_len2048"),
    os.path.join("..", "legacy_models", f"tokenized_train_data_{MAX_SAMPLES}_len{MAX_LEN}"),
    os.path.join("..", "legacy_models", "tokenized_train_data_10_len2048"),
    os.path.join("..", "legacy_models", "tokenized_train_data_100_len2048"),
]

train_data = None
for path in possible_paths:
    if os.path.exists(path):
        print(f"Found pre-tokenized dataset cache at: {path}")
        try:
            train_data = load_from_disk(path)
            if len(train_data) > MAX_SAMPLES:
                print(f"Selecting subset of {MAX_SAMPLES} samples from total {len(train_data)} samples.")
                train_data = train_data.select(range(MAX_SAMPLES))
            print("Dataset loaded successfully from disk cache!")
            break
        except Exception as e:
            print(f"Failed to load cached dataset: {e}")

if train_data is None:
    print("No cached dataset found on disk. Downloading and tokenizing on-the-fly...")
    train_data, _ = load_tulu_dataset("ai2-adapt-dev/flan_v2_converted", max_samples=MAX_SAMPLES)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.chat_template = CHAT_TEMPLATE
    preprocess_fn = get_byt5_preprocess_function(tokenizer, MAX_LEN, tokenizer.chat_template)

    train_data = train_data.map(
        preprocess_fn,
        batched=True,
        remove_columns=train_data.column_names,
        desc="Tokenizing train data"
    )

train_chat_dataset = MTPChatDataset(train_data)
train_loader = DataLoader(train_chat_dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"Loaded {len(train_chat_dataset)} samples.")

Setting up dataset...
Found pre-tokenized dataset cache at: ../saved_models/../legacy_models/tokenized_train_data_10_len2048
Dataset loaded successfully from disk cache!
Loaded 9 samples.


In [5]:
# Select target head class
if HEAD_TYPE.lower() == "cp":
    target_head_class = CanonicPolyidiac
elif HEAD_TYPE.lower() == "hmm":
    target_head_class = MTPC_HMM
else:
    raise ValueError(f"Unknown head type: {HEAD_TYPE}")

print(f"Instantiating MTP_LLM model with target head {target_head_class.__name__}...")
model = MTP_LLM(
    model_id=MODEL_ID,
    head_class=target_head_class,
    window_size=WINDOW_SIZE,
    ranks=RANKS,
    lora_path=lora_path,
    cheat=True
)
model.to(device)

print("Swapping head and transferring FF weights to target CP/HMM...")
swap_model_head(model, target_head_class, WINDOW_SIZE, RANKS, device)

# Load FF head state dict and initialize target head
ff_state_dict = torch.load(ff_head_path, map_location=device)
if target_head_class.__name__ == 'CanonicPolyidiac':
    init_cp_from_ff(model, ff_state_dict, WINDOW_SIZE, RANKS, device)
elif target_head_class.__name__ == 'MTPC_HMM':
    # HMM from FF initialization fallback to backbone STP
    from training import init_emissions_from_stp
    init_emissions_from_stp(model, WINDOW_SIZE, RANKS)

Instantiating MTP_LLM model with target head CanonicPolyidiac...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading LoRA adapter from: ../saved_models/lora_ff_w6_phase1_v1 (is_trainable=True)


Swapping head and transferring FF weights to target CP/HMM...


In [6]:
print("Evaluating initial loss before training...")
model.eval()
total_loss = 0.0
num_batches = 0
is_log_probs = target_head_class.__name__ in ['CanonicPolyidiac', 'MTPC_HMM']

with torch.no_grad():
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        _, mtp_logits, _ = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = compute_mtpc_loss(mtp_logits, labels, WINDOW_SIZE, gamma=0.8, is_log_probs=is_log_probs)
        
        total_loss += loss.item()
        num_batches += 1

initial_avg_loss = total_loss / num_batches
print(f"Average Initial Loss: {initial_avg_loss:.4f}")
assert initial_avg_loss < 10.0, f"Expected initial loss < 10, got {initial_avg_loss}"
print("SUCCESS: Initial loss is under 10!")

Evaluating initial loss before training...


Average Initial Loss: 3.1822
SUCCESS: Initial loss is under 10!


In [7]:
print("\n--- Starting Phase 2 Fine-Tuning Simulation ---")
model.backbone.train()
model.heads.train()

# Differential learning rates
param_groups = [
    {"params": list(model.heads.parameters()), "lr": LR_HEAD},
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": LR_BACKBONE}
]
optimizer = Adam(param_groups)

step_idx = 0
for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    pbar = tqdm(train_loader)
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        # Linear warmup for backbone LoRA parameter group
        if step_idx < WARMUP_STEPS:
            current_lora_lr = LR_BACKBONE * (step_idx / WARMUP_STEPS)
            optimizer.param_groups[1]['lr'] = current_lora_lr
        else:
            optimizer.param_groups[1]['lr'] = LR_BACKBONE
            
        optimizer.zero_grad()
        _, mtp_logits, _ = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = compute_mtpc_loss(mtp_logits, labels, WINDOW_SIZE, gamma=0.8, is_log_probs=is_log_probs)
        
        loss.backward()
        optimizer.step()
        
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr_lora=f"{optimizer.param_groups[1]['lr']:.2e}")
        step_idx += 1

print("Phase 2 training simulation finished successfully!")


--- Starting Phase 2 Fine-Tuning Simulation ---
Epoch 1/1


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:03<?, ?it/s, loss=5.8029, lr_lora=0.00e+00]

 20%|██        | 1/5 [00:03<00:15,  3.84s/it, loss=5.8029, lr_lora=0.00e+00]

 20%|██        | 1/5 [00:06<00:15,  3.84s/it, loss=5.5593, lr_lora=4.00e-06]

...

 80%|████████  | 4/5 [00:12<00:02,  2.88s/it, loss=5.7352, lr_lora=1.20e-05]

 80%|████████  | 4/5 [00:13<00:02,  2.88s/it, loss=6.0459, lr_lora=1.60e-05]

100%|██████████| 5/5 [00:13<00:00,  2.35s/it, loss=6.0459, lr_lora=1.60e-05]

100%|██████████| 5/5 [00:13<00:00,  2.68s/it, loss=6.0459, lr_lora=1.60e-05]

Phase 2 training simulation finished successfully!


In [8]:
# Save weights to models_dir
os.makedirs(MODELS_DIR, exist_ok=True)
head_name_str = HEAD_TYPE.lower()
save_weights_path = os.path.join(MODELS_DIR, f"mtp_head_{head_name_str}_w{WINDOW_SIZE}_phase2_test.pth")
torch.save(model.heads.state_dict(), save_weights_path)
print(f"Saved fine-tuned head weights to: {save_weights_path}")

lora_save_path = os.path.join(MODELS_DIR, f"lora_{head_name_str}_w{WINDOW_SIZE}_phase2_test")
model.backbone.save_pretrained(lora_save_path)
print(f"Saved fine-tuned LoRA backbone adapter to: {lora_save_path}")

Saved fine-tuned head weights to: ../saved_models/mtp_head_cp_w6_phase2_test.pth


Saved fine-tuned LoRA backbone adapter to: ../saved_models/lora_cp_w6_phase2_test
